# RAG and Agentic Search — Study Notes

## 1. Retrieval Augmented Generation (RAG)

### (1) Main idea
RAG is a technique for working with documents that are too big to fit into a single prompt. Instead of cramming everything into one massive prompt, RAG **breaks documents into chunks** and only includes the most **relevant** pieces when answering a question.

### (2) Two options for getting large-doc info into a prompt
1. **Option 1**: Include everything in the prompt (doesn't scale — context limits, cost, quality degradation)
2. **Option 2**: Break documents into chunks and retrieve only the relevant ones (this is RAG)

### (3) Benefits & Challenges

**Benefits**
- Claude focuses on only the most relevant content
- Scales up to very large documents
- Works across multiple documents
- Smaller prompts cost less and run faster

**Challenges**
- Requires a preprocessing step to chunk documents
- Needs a search mechanism to find "relevant" chunks
- Retrieved chunks might not contain all the context Claude needs
- Many ways to chunk text — which is best depends on the document type

### 💡 My understanding of where RAG sits
RAG happens **on the user side**, not the assistant side. The user's question gets compared against a large knowledge base, the most relevant chunks are pulled out, then combined with the original question into the prompt. From the LLM's perspective, it just sees a "user message that happens to come with a lot of context" — it has no idea retrieval even happened. RAG is a **developer-side preprocessing step** that solves three problems at once: **context window limits**, **API cost** (irrelevant tokens are wasted spend), and **quality** (LLMs suffer from "lost in the middle" with long, low-relevance context).

---

## 2. Text Chunking Strategies

### (1) Size-based chunking
- **Concept**: Divide text into strings of equal length. A 325-character document might become three chunks of ~108 characters each.
- **Pros**: Easy to implement; works with any document type.
- **Cons**: Words get cut mid-sentence; chunks lose surrounding context; section headers can be separated from their content.
- **Fix**: Add **overlap** between chunks — each chunk includes some characters from neighbors so words and sentences aren't truncated.
- *Code implementation below.*

### (2) Sentence-based chunking
- **Concept**: Split text into sentences with regex, then group sentences into chunks with optional overlap.
- **Pros**: Respects sentence boundaries; preserves more natural meaning than raw character splits.
- **Cons**: Sentence length varies wildly, so chunk size becomes uneven; regex sentence splitting struggles with abbreviations ("Dr.", "U.S.").
- *Code implementation below.*

### (3) Structure-based chunking
- **Concept**: Split by the document's natural structure — headers, paragraphs, sections. Works great for well-formatted Markdown.
- **Pros**: Cleanest, most semantically meaningful chunks — each chunk represents a complete section.
- **Cons**: Only works when you can guarantee structure. Most real-world plain text or PDFs lack clean structural markers.
- *Code implementation below.*

### (4) Semantic chunking
- **Concept**: Split into sentences, then use NLP to measure how related consecutive sentences are. Group related sentences into chunks.
- **Pros**: Produces the most relevant, topically-coherent chunks.
- **Cons**: Most sophisticated approach; computationally expensive (needs embeddings during preprocessing).

---

## 3. Text Embeddings

### (1) Why text embeddings
After chunking, the next problem is: **which chunks are most relevant to the user's question?** This is a search problem.

The dominant approach is **semantic search**. Unlike keyword search (exact word matches), semantic search uses embeddings to capture the **meaning** of both the user's question and each chunk, then compares them in vector space.

### (2) Embedding process
- Feed text into an embedding model
- The model outputs a long list of numbers (the embedding vector)
- Each number is in the range [-1, +1]
- Each dimension represents some latent semantic feature of the text

### (3) Implementation with VoyageAI
Anthropic doesn't provide embedding models. VoyageAI is the officially recommended provider, particularly strong on retrieval quality.

**Note on `input_type`**: VoyageAI's `embed()` accepts `input_type="query"` or `input_type="document"`. The server prepends a special prompt before embedding — `"Represent the query for retrieving supporting documents: "` or `"Represent the document for retrieval: "` — so the model produces vectors better tuned to the asymmetric query-vs-document retrieval task. Always set this for RAG. Use `None` for symmetric tasks like clustering or similarity comparison.

**Note on tokenization**: Tokenization still happens — just on Voyage's servers, transparent to you. The SDK is essentially an HTTP wrapper around `string in → vector out`. You can preview the tokens with `vo.tokenize(...)` or load the tokenizer locally via `AutoTokenizer.from_pretrained("voyageai/voyage-3-large")`. The fact that the API enforces per-request token limits and charges by token is proof the tokenization step is real.

---

## 4. The Full RAG Flow

### (1) Pipeline
1. **Chunk** the source text
2. **Generate embeddings** for each chunk (+ normalization)
3. **Store** vectors in a vector database (with the original chunk as metadata)
4. **Process user query** → embed it
5. **Find similar embeddings** → typically via cosine similarity
6. **Build the final prompt** → combine user's question + top-K relevant chunks → send to Claude

### (2) Why we store metadata alongside the vector
A vector by itself is just a list of floats — unreadable. You need the original chunk text to feed back into the LLM. So `add_vector(embedding, {"content": chunk, ...})` pairs each vector with its source text (and any other useful fields: source file, section title, timestamp, etc., for filtering and citations).

*Code implementation below.*

---

## 5. BM25 Lexical Search + Hybrid Retrieval

### (1) Why combine lexical search with semantic search
Semantic search alone misses exact-match cases. Things like product IDs, error codes, specific names, or rare technical terms (e.g., `"INC-2023-Q4-011"` or `"CLAUDE_CODE_EXPERIMENTAL_AGENT_TEAMS"`) need **exact term matching**, which embeddings don't reliably provide.

| | Semantic Search | BM25 (Lexical) |
|---|---|---|
| Matches on | Meaning / concepts | Exact keywords |
| Strong at | Synonyms, paraphrasing, multilingual | Proper nouns, codes, IDs, rare terms |
| Weak at | Exact string matching | Synonyms, conceptual relevance |

**Hybrid search** = combine both. Production RAG systems almost always do this.

### (2) How BM25 (Best Match 25) works
1. **Tokenize the query** → break the question into individual terms
2. **Count term frequency** → how often does each term appear in each document?
3. **Weight terms by inverse document frequency (IDF)** → rare terms get higher importance ("a" → low weight, "INC-2023-Q4-011" → high weight)
4. **Score & rank** → documents containing more high-weight terms (with appropriate length normalization) rank higher

### (3) BM25 code implementation
*See code below.*

### (4) Combining semantic + lexical (Hybrid Retrieval)
The standard fusion technique is **Reciprocal Rank Fusion (RRF)**:
- Run both retrievers independently, get top-K from each
- For each doc, score = Σ `1 / (k_rrf + rank_in_index_i)` across all indexes
- Sort by combined RRF score, return top-K

RRF is rank-based, not score-based, so it doesn't care that BM25 and semantic search produce scores on incompatible scales — it just cares about ordering.

*Code implementation below.*

---

## 6. From Traditional RAG to Agentic RAG

### (1) The shift
**Traditional RAG** is a fixed, one-shot pipeline:

```
user query → embed → retrieve once → stuff into prompt → generate answer
```

The LLM is **passive** — it only joins at the end. Retrieval logic is hard-coded.

**Agentic Search** wraps retrieval as a **tool** and lets the LLM decide how to use it:

```
LLM decides: "Do I need to search? What query? How many times? Which index?"
   ↓ calls tool
retrieval runs
   ↓ returns chunks
LLM evaluates: "Enough? Or search again with a different query?"
   ↓ (loop if needed)
LLM generates final answer
```

The LLM is now the **decision-maker driving the retrieval flow**, not just the answerer.

### (2) What "agentic" means in practice
With retrieval exposed as a tool, the LLM can autonomously:
- **Decide whether to retrieve** — skip retrieval for "what's today's date", trigger it for "what did Q3 report say"
- **Rewrite queries** — turn casual phrasing into retrieval-optimized keywords
- **Multi-hop retrieval** — break complex questions into sub-questions, search each separately
- **Pick the right index** — finance Q → finance docs; code Q → codebase
- **Evaluate result quality** — retry with a different query if results look weak
- **Combine multiple tools** — internal search + web search + calculator, then synthesize

### (3) Layered architecture
Agentic search doesn't replace traditional RAG — it sits **on top of it**:

```
Agentic Layer:    LLM decides when/what/how-many-times to search
   ↓ calls
Retrieval Layer:  Hybrid Search (Embedding + BM25 + Reranker)
   ↓ operates on
Index Layer:      Vector DB + BM25 Index + Metadata
   ↓ stores
Data Layer:       Chunks of original documents
```

Agentic is the **commander** at the top; all the retrieval techniques (chunking, embedding, BM25, multi-index, reranking) are **the weapons** it commands.

### 💡 One-line takeaway
**Traditional RAG**: the developer fetches context for the LLM.
**Agentic RAG**: the LLM fetches its own context by using retrieval as a tool.
The shift is from **code-driven control flow** to **LLM-driven control flow**. The tradeoff is more LLM calls (higher latency + cost) in exchange for handling complex, multi-hop, or cross-source questions that one-shot RAG can't.


In [1]:
import re

In [2]:
## 2. Text chuncking strategy
# (1) Size based chunking - code implementation
def chunk_by_char(text, chunk_size=150, chunk_overlap=20):
    start_idx = 0
    chunks = []

    while start_idx < len(text):
        end_idx = min(start_idx + chunk_size, len(text))
        chunks.append(text[start_idx:end_idx])

        start_idx = end_idx - chunk_overlap if end_idx < len(text) else len(text) 

    return chunks

In [ ]:
# (2) Sentence based chunking - code implementation
def chunk_by_sentence(text, max_sentences_per_chunk=5, overlap_sentences=1):
    sentences = re.split(r"(?<=[.!?])\s+", text)
    
    chunks = []
    start_idx = 0
    
    while start_idx < len(sentences):
        end_idx = min(start_idx + max_sentences_per_chunk, len(sentences))
        current_chunk = sentences[start_idx:end_idx]
        chunks.append(" ".join(current_chunk))
        
        start_idx += max_sentences_per_chunk - overlap_sentences
        
        if start_idx < 0:
            start_idx = 0
    
    return chunks

In [4]:
# (3) Structure based chunking - code implementation
def chunk_by_section(document_text):
    pattern = r"\n## "
    return re.split(pattern, document_text)

In [ ]:
## 3. Text embedding - Implementation with VoyageAI
from dotenv import load_dotenv
import voyageai

load_dotenv()
client = voyageai.Client()


/Users/ionahu/sources/NomNom/learning_lab/.venv/bin/python: No module named pip
Note: you may need to restart the kernel to use updated packages.


In [36]:
from dotenv import load_dotenv
import voyageai

load_dotenv()
client = voyageai.Client()

# note: input_type can be either "query" or "document"
def generate_embedding(chunks, model="voyage-3-large", input_type="query"):

    is_list = isinstance(chunks, list)
    input = chunks if is_list else [chunks]
    result = client.embed(input, model=model, input_type=input_type)
    return result.embeddings if is_list else result.embeddings[0]

In [ ]:
with open(
    "/Users/ionahu/sources/NomNom/docs/learning/05_learning_notes/01_agent_patterns_workflows.md", "r"
) as f:
    text = f.read()

chunks = chunk_by_section(text)

generate_embedding(chunks[0])

[-0.08173003792762756,
 0.04605972021818161,
 -0.011688088066875935,
 0.009220602922141552,
 0.0069695631973445415,
 0.010432700626552105,
 -0.012986763380467892,
 0.012640451081097126,
 -0.027878252789378166,
 0.01601700857281685,
 -0.047791291028261185,
 0.03584346920251846,
 0.004610301461070776,
 -0.05160073935985565,
 -0.032207176089286804,
 -0.012380714528262615,
 0.05367862433195114,
 0.09558258205652237,
 -0.04536709561944008,
 -0.05818070471286774,
 0.04103817418217659,
 -0.05437125265598297,
 -0.01419886201620102,
 -0.04709866642951965,
 -0.00744574423879385,
 -0.020951978862285614,
 0.038440823554992676,
 0.06579960882663727,
 -0.010129676200449467,
 -0.014545176178216934,
 -0.027878252789378166,
 -0.002186105353757739,
 -0.02337617613375187,
 0.00974007323384285,
 -0.012294136919081211,
 0.040691860020160675,
 -0.021211715415120125,
 -0.021038556471467018,
 -0.028570881113409996,
 -0.03030245006084442,
 -0.004761813208460808,
 -0.005389506928622723,
 0.025800369679927826,
 

In [37]:
# ## 4. The full RAG flow
# - (1) Flow
# Step 1: Chunk Your Source Text
with open(
    "/Users/ionahu/sources/NomNom/docs/learning/05_learning_notes/01_agent_patterns_workflows.md", "r"
) as f:
    text = f.read()

chunks = chunk_by_section(text)
print(chunks[2])  # Test to see the table of contents
print(len(chunks))  # Test to see the number of chunks

1. The Three Layers of Agentic Systems

"Agentic system" 是一个伞状术语,涵盖从增强单次调用到完全自主 agent 的所有形式。三层,复杂度递增:

| Layer | Essence | Control |
|---|---|---|
| **Augmented LLM** | Enhanced single LLM call | Fully developer-controlled |
| **Workflow** | Multiple LLM calls along predefined code paths | Path hardcoded in advance |
| **Agent** | LLM plans, uses tools, and loops autonomously | LLM makes dynamic decisions |

### Key distinction (Workflow vs Agent)

- **Workflow**: Path is written into code by the developer → **LLM 是执行者**
- **Agent**: Path is decided by the LLM itself → **LLM 是决策者**

---

12


In [40]:
# Step 2: Generate Embeddings + Normalization
embeddings = generate_embedding(chunks)
print(len(embeddings))
print(len(embeddings[0]))

12
1024


In [41]:
# Step 3: Store in Vector Database
# VectorIndex implementation
import math
from typing import Optional, Any, List, Dict, Tuple


class VectorIndex:
    def __init__(
        self,
        distance_metric: str = "cosine",
        embedding_fn=None,
    ):
        self.vectors: List[List[float]] = []
        self.documents: List[Dict[str, Any]] = []
        self._vector_dim: Optional[int] = None
        if distance_metric not in ["cosine", "euclidean"]:
            raise ValueError("distance_metric must be 'cosine' or 'euclidean'")
        self._distance_metric = distance_metric
        self._embedding_fn = embedding_fn

    def add_document(self, document: Dict[str, Any]):
        if not self._embedding_fn:
            raise ValueError(
                "Embedding function not provided during initialization."
            )
        if not isinstance(document, dict):
            raise TypeError("Document must be a dictionary.")
        if "content" not in document:
            raise ValueError(
                "Document dictionary must contain a 'content' key."
            )

        content = document["content"]
        if not isinstance(content, str):
            raise TypeError("Document 'content' must be a string.")

        vector = self._embedding_fn(content)
        self.add_vector(vector=vector, document=document)

    def search(
        self, query: Any, k: int = 1
    ) -> List[Tuple[Dict[str, Any], float]]:
        if not self.vectors:
            return []

        if isinstance(query, str):
            if not self._embedding_fn:
                raise ValueError(
                    "Embedding function not provided for string query."
                )
            query_vector = self._embedding_fn(query)
        elif isinstance(query, list) and all(
            isinstance(x, (int, float)) for x in query
        ):
            query_vector = query
        else:
            raise TypeError(
                "Query must be either a string or a list of numbers."
            )

        if self._vector_dim is None:
            return []

        if len(query_vector) != self._vector_dim:
            raise ValueError(
                f"Query vector dimension mismatch. Expected {self._vector_dim}, got {len(query_vector)}"
            )

        if k <= 0:
            raise ValueError("k must be a positive integer.")

        if self._distance_metric == "cosine":
            dist_func = self._cosine_distance
        else:
            dist_func = self._euclidean_distance

        distances = []
        for i, stored_vector in enumerate(self.vectors):
            distance = dist_func(query_vector, stored_vector)
            distances.append((distance, self.documents[i]))

        distances.sort(key=lambda item: item[0])

        return [(doc, dist) for dist, doc in distances[:k]]

    def add_vector(self, vector, document: Dict[str, Any]):
        if not isinstance(vector, list) or not all(
            isinstance(x, (int, float)) for x in vector
        ):
            raise TypeError("Vector must be a list of numbers.")
        if not isinstance(document, dict):
            raise TypeError("Document must be a dictionary.")
        if "content" not in document:
            raise ValueError(
                "Document dictionary must contain a 'content' key."
            )

        if not self.vectors:
            self._vector_dim = len(vector)
        elif len(vector) != self._vector_dim:
            raise ValueError(
                f"Inconsistent vector dimension. Expected {self._vector_dim}, got {len(vector)}"
            )

        self.vectors.append(list(vector))
        self.documents.append(document)

    def _euclidean_distance(
        self, vec1: List[float], vec2: List[float]
    ) -> float:
        if len(vec1) != len(vec2):
            raise ValueError("Vectors must have the same dimension")
        return math.sqrt(sum((p - q) ** 2 for p, q in zip(vec1, vec2)))

    def _dot_product(self, vec1: List[float], vec2: List[float]) -> float:
        if len(vec1) != len(vec2):
            raise ValueError("Vectors must have the same dimension")
        return sum(p * q for p, q in zip(vec1, vec2))

    def _magnitude(self, vec: List[float]) -> float:
        return math.sqrt(sum(x * x for x in vec))

    def _cosine_distance(self, vec1: List[float], vec2: List[float]) -> float:
        if len(vec1) != len(vec2):
            raise ValueError("Vectors must have the same dimension")

        mag1 = self._magnitude(vec1)
        mag2 = self._magnitude(vec2)

        if mag1 == 0 and mag2 == 0:
            return 0.0
        elif mag1 == 0 or mag2 == 0:
            return 1.0

        dot_prod = self._dot_product(vec1, vec2)
        cosine_similarity = dot_prod / (mag1 * mag2)
        cosine_similarity = max(-1.0, min(1.0, cosine_similarity))

        return 1.0 - cosine_similarity

    def __len__(self) -> int:
        return len(self.vectors)

    def __repr__(self) -> str:
        has_embed_fn = "Yes" if self._embedding_fn else "No"
        return f"VectorIndex(count={len(self)}, dim={self._vector_dim}, metric='{self._distance_metric}', has_embedding_fn='{has_embed_fn}')"


In [42]:
store = VectorIndex()

for embedding, chunk in zip(embeddings, chunks):
    store.add_vector(embedding, {"content":chunk})

In [43]:
# Step 4: Process User Query
user_embedding = generate_embedding("How different are agent and workflow? when to use which")

In [44]:
# Step 5: Find Similar Embeddings -->  Based on Cosine Similarity
results = store.search(user_embedding, 2)

for doc, distance in results:
    print(distance, "\n", doc["content"][0:200], "\n")

0.35206228542865814 
 6. Key Takeaways

1. **别默认上 agent** — 大多数应用用一个优化好的单次 LLM 调用 + RAG + few-shot 就够了
2. **Workflow 和 agent 不是替代关系** — 它们是不同复杂度的工具,按需选择
3. **谨慎使用框架** — 先掌握底层 API,再考虑抽象层
4. **通过测量来迭代** — 任何复杂度增加都必须被评估证明合理
5 

0.35254666159390313 
 1. The Three Layers of Agentic Systems

"Agentic system" 是一个伞状术语,涵盖从增强单次调用到完全自主 agent 的所有形式。三层,复杂度递增:

| Layer | Essence | Control |
|---|---|---|
| **Augmented LLM** | Enhanced single LLM call | Full 



In [46]:
## 5. BM25 Lexical Search - Code implementation
# Chunk by section
import re


def chunk_by_section(document_text):
    pattern = r"\n## "
    return re.split(pattern, document_text)



In [48]:
# BM25Index implementation
import math
from collections import Counter
from typing import Callable, Optional, Any, List, Dict, Tuple


class BM25Index:
    def __init__(
        self,
        k1: float = 1.5,
        b: float = 0.75,
        tokenizer: Optional[Callable[[str], List[str]]] = None,
    ):
        self.documents: List[Dict[str, Any]] = []
        self._corpus_tokens: List[List[str]] = []
        self._doc_len: List[int] = []
        self._doc_freqs: Dict[str, int] = {}
        self._avg_doc_len: float = 0.0
        self._idf: Dict[str, float] = {}
        self._index_built: bool = False

        self.k1 = k1
        self.b = b
        self._tokenizer = tokenizer if tokenizer else self._default_tokenizer

    def _default_tokenizer(self, text: str) -> List[str]:
        text = text.lower()
        tokens = re.split(r"\W+", text)
        return [token for token in tokens if token]

    def _update_stats_add(self, doc_tokens: List[str]):
        self._doc_len.append(len(doc_tokens))

        seen_in_doc = set()
        for token in doc_tokens:
            if token not in seen_in_doc:
                self._doc_freqs[token] = self._doc_freqs.get(token, 0) + 1
                seen_in_doc.add(token)

        self._index_built = False

    def _calculate_idf(self):
        N = len(self.documents)
        self._idf = {}
        for term, freq in self._doc_freqs.items():
            idf_score = math.log(((N - freq + 0.5) / (freq + 0.5)) + 1)
            self._idf[term] = idf_score

    def _build_index(self):
        if not self.documents:
            self._avg_doc_len = 0.0
            self._idf = {}
            self._index_built = True
            return

        self._avg_doc_len = sum(self._doc_len) / len(self.documents)
        self._calculate_idf()
        self._index_built = True

    def add_document(self, document: Dict[str, Any]):
        if not isinstance(document, dict):
            raise TypeError("Document must be a dictionary.")
        if "content" not in document:
            raise ValueError(
                "Document dictionary must contain a 'content' key."
            )

        content = document.get("content", "")
        if not isinstance(content, str):
            raise TypeError("Document 'content' must be a string.")

        doc_tokens = self._tokenizer(content)

        self.documents.append(document)
        self._corpus_tokens.append(doc_tokens)
        self._update_stats_add(doc_tokens)

    def _compute_bm25_score(
        self, query_tokens: List[str], doc_index: int
    ) -> float:
        score = 0.0
        doc_term_counts = Counter(self._corpus_tokens[doc_index])
        doc_length = self._doc_len[doc_index]

        for token in query_tokens:
            if token not in self._idf:
                continue

            idf = self._idf[token]
            term_freq = doc_term_counts.get(token, 0)

            numerator = idf * term_freq * (self.k1 + 1)
            denominator = term_freq + self.k1 * (
                1 - self.b + self.b * (doc_length / self._avg_doc_len)
            )
            score += numerator / (denominator + 1e-9)

        return score

    def search(
        self,
        query_text: str,
        k: int = 1,
        score_normalization_factor: float = 0.1,
    ) -> List[Tuple[Dict[str, Any], float]]:
        if not self.documents:
            return []

        if not isinstance(query_text, str):
            raise TypeError("Query text must be a string.")

        if k <= 0:
            raise ValueError("k must be a positive integer.")

        if not self._index_built:
            self._build_index()

        if self._avg_doc_len == 0:
            return []

        query_tokens = self._tokenizer(query_text)
        if not query_tokens:
            return []

        raw_scores = []
        for i in range(len(self.documents)):
            raw_score = self._compute_bm25_score(query_tokens, i)
            if raw_score > 1e-9:
                raw_scores.append((raw_score, self.documents[i]))

        raw_scores.sort(key=lambda item: item[0], reverse=True)

        normalized_results = []
        for raw_score, doc in raw_scores[:k]:
            normalized_score = math.exp(-score_normalization_factor * raw_score)
            normalized_results.append((doc, normalized_score))

        normalized_results.sort(key=lambda item: item[1])

        return normalized_results

    def __len__(self) -> int:
        return len(self.documents)

    def __repr__(self) -> str:
        return f"BM25VectorStore(count={len(self)}, k1={self.k1}, b={self.b}, index_built={self._index_built})"

In [49]:
# 1. Chunk your text by sections
chunks = chunk_by_section(text)

# 2. Create a BM25 store and add documents
store = BM25Index()
for chunk in chunks:
    store.add_document({"content": chunk})

# 3. Search the store
results = store.search("What happened with INC-2023-Q4-011?", 3)

# Print results
for doc, distance in results:
    print(distance, "\n", doc["content"][:200], "\n----\n")

# BM25Index implementation
import math
from collections import Counter
from typing import Callable, Optional, Any, List, Dict, Tuple



0.5585868277253188 
 9. Interview Q&A

### 🟢 Level 1: Foundational Understanding

1. What's the difference between a workflow and an agent according to Anthropic's definition?
2. What are the three components that make up 
----

0.8050767891296614 
 2. How to Choose (Decision Framework)

### Core Principle

> Start with the simplest solution; add complexity only when it demonstrably improves outcomes.

### Increasing Complexity Ladder

```
Single 
----



In [ ]:
## 5. BM25 Lexical search combined with Semantic seach
# (4) Implementation: Combine Semantic + lexical seach
class Retriever:
    def __init__(self, *indexes: SearchIndex):
        if len(indexes) == 0:
            raise ValueError("At least one index must be provided")
        self._indexes = list(indexes)
    
    def add_document(self, document: Dict[str, Any]):
        for index in self._indexes:
            index.add_document(document)
    
    def search(self, query_text: str, k: int = 1, k_rrf: int = 60):
        # Get results from all indexes
        all_results = []
        for idx, results in enumerate(all_results):
            for rank, (doc, _) in enumerate(results):
                # Track document ranks across indexes
                # Apply RRF scoring formula
        # Return merged and sorted results

# Interview Q&A — RAG & Agentic Search

Common questions for MLE / AI Engineer / AI Researcher interviews, with answers framed for the way I'd actually say them out loud.

---

### Q1: What is RAG and why do we need it?

RAG (Retrieval-Augmented Generation) is a technique where, instead of stuffing an entire knowledge base into the prompt, we **retrieve only the most relevant chunks** and combine them with the user's question before sending it to the LLM.

We need it because:
1. **Context windows are finite** — even at 200K tokens we can't hold an entire knowledge base
2. **Cost** — every irrelevant token is wasted API spend
3. **Quality** — LLMs suffer from "lost in the middle" with long, low-signal context

RAG is a developer-side preprocessing step. The LLM doesn't know retrieval happened — it just sees a user message that happens to come with context.

---

### Q2: Walk me through a typical RAG pipeline.

Six steps:
1. **Chunk** the source documents (size-based, sentence-based, structure-based, or semantic)
2. **Embed** each chunk with an embedding model (e.g., VoyageAI `voyage-3-large` with `input_type="document"`)
3. **Index** the vectors in a vector store, paired with metadata (original text, source, section, timestamps)
4. **Embed the user query** (with `input_type="query"`)
5. **Retrieve** top-K most similar chunks via cosine similarity (or hybrid with BM25)
6. **Generate** — assemble final prompt with retrieved chunks + user question, send to the LLM

---

### Q3: What are the different chunking strategies and tradeoffs?

| Strategy | Pros | Cons |
|---|---|---|
| Size-based | Simple, works on any text | Cuts mid-sentence; loses context |
| Sentence-based | Respects sentence boundaries | Uneven chunk sizes; regex fragile |
| Structure-based | Semantically clean (best for Markdown) | Requires structured documents |
| Semantic | Most topically coherent chunks | Computationally expensive |

In production I'd start with structure-based for Markdown/HTML, fall back to sentence-based with overlap for plain text, and only reach for semantic chunking if retrieval quality is clearly the bottleneck.

---

### Q4: How does an embedding model turn text into a vector? Where does tokenization happen when using an API?

The model first **tokenizes** the text (subword units, typically BPE), maps tokens to embedding vectors, runs them through a transformer encoder, and pools the output (mean, [CLS], or attention pooling) into a single fixed-dimensional vector.

When using an API like VoyageAI, tokenization happens **on the provider's servers** — the client SDK is just an HTTP wrapper around `string in → vector out`. You can verify the tokenization exists because the API enforces per-request token limits, charges by token, and exposes `count_tokens()` and `tokenize()` helper methods. Voyage even open-sources its tokenizers on Hugging Face if you want to inspect them locally.

---

### Q5: Why does VoyageAI use `input_type="query"` vs `input_type="document"`?

Retrieval is an **asymmetric task** — queries are usually short and interrogative, documents are long and declarative. Voyage's models are trained to handle these two roles differently. Setting `input_type` causes the server to **prepend a role-specific prompt** before embedding (e.g., `"Represent the query for retrieving supporting documents: "`), which produces vectors better tuned to retrieval. The two are compatible in the same vector space.

For symmetric tasks (clustering, similarity comparison between equal-status texts), use `None`.

---

### Q6: Cosine similarity vs Euclidean distance for embedding retrieval — which and why?

**Cosine similarity** is the standard choice. It measures the angle between vectors, ignoring magnitude. Two important practical points:
- Most embedding models (Voyage, OpenAI) return **L2-normalized vectors**. When vectors are unit-length, cosine similarity equals dot product, and rankings under cosine, dot-product, and Euclidean become identical.
- Cosine is preferred conceptually because **semantic similarity is direction-based** — a longer vector doesn't mean "more meaning", just a different magnitude artifact.

---

### Q7: What is BM25 and why combine it with semantic search?

BM25 is a classical **lexical retrieval algorithm** based on term frequency and inverse document frequency (with length normalization). It's the modern descendant of TF-IDF.

Why combine it with semantic search:
- Semantic search excels at **synonyms and paraphrasing**, but is bad at **exact-match cases** — product IDs, error codes, proper nouns, rare technical terms
- BM25 excels at exact matching but is blind to synonyms
- Combining them ("hybrid search") gets the best of both

The standard combination technique is **Reciprocal Rank Fusion (RRF)** — rank-based aggregation that's robust to differing score scales.

---

### Q8: What's the difference between RAG and Agentic RAG?

**Traditional RAG** is a fixed pipeline — embed query, retrieve top-K, stuff into prompt, generate. The LLM is passive and only participates at the very end. Retrieval is controlled by code.

**Agentic RAG** wraps retrieval as a **tool**. The LLM autonomously decides:
- Whether to search at all
- What query string to use (it can rewrite the user's question)
- How many times to search (multi-hop reasoning)
- Which index/source to search
- Whether the results are good enough or it needs to retry

It's a shift from **code-driven control flow** to **LLM-driven control flow**. The tradeoff is **higher latency and cost** (multiple LLM round-trips per answer) in exchange for handling **complex, multi-hop, or cross-source questions** that one-shot RAG can't.

Architecturally, agentic sits **on top of** hybrid retrieval — it doesn't replace it.

---

### Q9: How would you evaluate a RAG system?

I'd separate evaluation into two layers:

**Retrieval evaluation** (does retrieval find the right chunks?):
- Recall@K — does the gold chunk show up in top-K?
- MRR (Mean Reciprocal Rank) — how high is the gold chunk ranked?
- NDCG@K — for graded relevance, captures ranking quality
- Need a labeled query → gold chunk evaluation set

**Generation evaluation** (does the LLM produce a good answer?):
- Faithfulness / groundedness — is the answer supported by the retrieved chunks? (LLM-as-judge or claim-level verification)
- Answer relevance — does the answer address the question?
- Citation accuracy — if the system claims a source, is it really from there?

Tools like RAGAS or TruLens are common for the generation side.

---

### Q10: What are common production failures in RAG and how would you address them?

| Failure | Cause | Mitigation |
|---|---|---|
| Retrieves irrelevant chunks | Bad chunking (cuts context) or weak embedding model | Better chunking strategy, try a stronger model (Voyage / Cohere), add reranker |
| Misses exact-term queries | Pure semantic search | Add BM25, hybrid fusion |
| Retrieves correct chunk but LLM ignores it | "Lost in the middle" or low-quality prompt template | Use a reranker (e.g., Voyage `rerank-2.5`), reorder context, shorten context window |
| Stale / outdated answers | Static index | Add freshness signals, incremental indexing, source filtering by timestamp |
| Hallucinated answers despite good retrieval | LLM not anchored to retrieved context | Stronger prompt instructions ("answer only from the context below"), citation-required output, faithfulness eval in CI |
| One-shot RAG fails on complex queries | Single retrieval pass | Move to agentic RAG with tool-use loops |

---

### Q11: How would you handle a very large knowledge base (millions of documents)?

Several considerations:
- **Vector store choice**: FAISS for local/single-machine, Pinecone/Weaviate/Qdrant for distributed
- **ANN indexing**: HNSW or IVF-PQ instead of exact search — sub-linear query time at the cost of small recall loss
- **Sharding by metadata**: split by tenant, language, time period, or domain so each query hits a smaller index
- **Multi-stage retrieval**: cheap BM25 first-pass (filter to top-1000), then expensive embedding similarity, then a reranker for the final top-K
- **Caching**: cache popular queries and their retrieved chunks
- **Async batched embedding**: amortize API calls when indexing

---

### Q12: What's a reranker and when do you use one?

A reranker is a **second-stage model** that takes the top-K results from initial retrieval (say, top-100) and reorders them to surface the truly best ones (top-5). Rerankers are usually **cross-encoders** — they take `(query, document)` as a pair and output a relevance score. They're more accurate than embedding-based bi-encoders but too slow to run over the entire corpus, which is why they're used as a second stage.

Voyage's `rerank-2.5`, Cohere's `rerank-v3`, and open-source models like `bge-reranker` are common choices.

Use a reranker when:
- Retrieval recall is fine but ranking is mediocre
- You need very high precision in the top few results
- You're willing to add latency (rerankers add ~100-300ms)
